In [2]:
import torch
from predictors.goal_probability_predictor import WorldCupGoalsPredictor
import json
from typing import List
import pickle
import pandas as pd

In [29]:
with open("models/6_goals_softmax/configs.json", "r") as f:
    cfg = json.load(f)

with open("preprocessed_data/2026_featurised_squads.json", "r") as f:
    squad_features = json.load(f)

model = WorldCupGoalsPredictor(cfg)
state_dict = torch.load("models/6_goals_softmax/model.pt", map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

scaler = pickle.load(open("models/6_goals_softmax/scalers.pkl", "rb"))

In [30]:
def featurise(team1: str, team2: str, feature_order_list: List[str], isknockout: int=0) -> List[float]:
    team_1_features = squad_features[team1]
    team_2_features = squad_features[team2]
    features = {}
    for f in feature_order_list:
        if f == "is_knockout":
            features[f] = float(isknockout)
            continue

        if "team_1" in f:
            features[f] = float(team_1_features[f.replace("team_1_", "")])
        elif "team_2" in f:
            features[f] = float(team_2_features[f.replace("team_2_", "")])
        else:
            raise NotImplementedError(f"featurise function needs to be updated to handle feature {f}")

    return features
        

def predict_goals(team1, team2, isknockout=0, temperature=0.8):
    team_1 = featurise(team1, team2, cfg["model"]["features"], isknockout)
    team_2 = featurise(team2, team1, cfg["model"]["features"], isknockout)
    data = pd.DataFrame([team_1, team_2])
    data_scaled = scaler.transform(data)
    data_tensor = torch.tensor(data_scaled, dtype=torch.float32)
    with torch.no_grad():
        res = model.predict_with_temperature(data_tensor, temperature=temperature)
    return res


In [48]:
predictions = {}
team_1_wins = 0
draws = 0
team_2_wins = 0
for i in range(100):
    res = predict_goals("England", "Spain", isknockout=0, temperature=0.45)
    t1 = int(res[0])
    t2 = int(res[1])
    res_tuple = (t1, t2)
    if res_tuple not in predictions:
        predictions[res_tuple] = 1
    else:
        predictions[res_tuple] += 1

    if t1 > t2:
        team_1_wins += 1
    elif t1 == t2:
        draws += 1
    else:
        team_2_wins += 1

print(f"Team 1 wins: {team_1_wins}")
print(f"Draws: {draws}")
print(f"Team 2 wins: {team_2_wins}")

predictions


Team 1 wins: 17
Draws: 48
Team 2 wins: 35


{(0, 0): 40,
 (1, 1): 6,
 (1, 0): 14,
 (1, 2): 5,
 (0, 1): 10,
 (0, 2): 20,
 (2, 0): 3,
 (2, 2): 2}

In [32]:
model.input_dim

31